# series/generative-adversarial-networks/02 GAN Training

Part 2 of the GAN series. Fresh CPU kernel; downloads official MNIST. Complete source batch-update examples, repaired missing setup, image/label variable collisions, undefined optimizer and self reference. These are batch update demonstrations, not a 9,500-epoch training run; the historical source figure is not reproduced by these few updates. All original code remains in appendix.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "tensorflow==2.20.0", "keras==3.15.1", "numpy==2.4.6", "certifi==2026.10.23"], check=True)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Sequential, Input
from tensorflow.keras.layers import Conv2D, LeakyReLU, Dropout, ZeroPadding2D, BatchNormalization, Flatten, Dense, Reshape, UpSampling2D
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
keras.utils.set_random_seed(42)
np.set_printoptions(precision=6, suppress=True)
print("Python:", sys.version.split()[0], "| TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

## Model Definitions

In [ ]:
def discriminator_model():
    discriminator = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, kernel_size=3, strides=2, padding="same"),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Conv2D(64, kernel_size=3, strides=2, padding="same"),
        ZeroPadding2D(padding=((0, 1), (0, 1))),
        BatchNormalization(momentum=0.8),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Conv2D(128, kernel_size=3, strides=2, padding="same"),
        BatchNormalization(momentum=0.8),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Conv2D(256, kernel_size=3, strides=1, padding="same"),
        BatchNormalization(momentum=0.8),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Flatten(), Dense(1, activation="sigmoid"),
    ], name="discriminator")
    return discriminator

def generator_model():
    generator = Sequential([
        Input(shape=(100,)),
        Dense(128 * 7 * 7, activation="relu"),
        Reshape((7, 7, 128)), UpSampling2D(size=(2, 2)),
        Conv2D(128, kernel_size=3, padding="same", activation="relu"),
        BatchNormalization(momentum=0.8),
        UpSampling2D(size=(2, 2)),
        Conv2D(64, kernel_size=3, padding="same", activation="relu"),
        BatchNormalization(momentum=0.8),
        Conv2D(1, kernel_size=3, padding="same", activation="relu"),
    ], name="generator")
    return generator

print("Both source network factories are ready.")

## Example 0

In [ ]:
discriminator = discriminator_model()
discriminator.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(), metrics=["accuracy"])
generator = generator_model()
(mnist_images, _), _ = keras.datasets.mnist.load_data()
batch_size = 32
imgs = mnist_images[:batch_size].astype("float32")[..., None] / 255.0
real_labels = np.ones((batch_size, 1), dtype="float32")
fake_labels = np.zeros((batch_size, 1), dtype="float32")
print("MNIST training records:", len(mnist_images), "| real batch:", imgs.shape)
print("Discriminator optimizer:", type(discriminator.optimizer).__name__)

## Example 1

In [ ]:
noise = np.random.normal(0, 1, (batch_size, 100)).astype("float32")
gen_imgs = generator(noise, training=False).numpy()
generator_before_d = [v.numpy().copy() for v in generator.weights]
discriminator_before_d = [v.numpy().copy() for v in discriminator.weights]
discriminator.reset_metrics()
d_loss_real = discriminator.train_on_batch(imgs, real_labels)
discriminator.reset_metrics()
d_loss_fake = discriminator.train_on_batch(gen_imgs, fake_labels)
print("Real batch loss / accuracy:", np.asarray(d_loss_real))
print("Fake batch loss / accuracy:", np.asarray(d_loss_fake))
assert all(np.array_equal(v.numpy(), old) for v, old in zip(generator.weights, generator_before_d))
assert any(not np.array_equal(v.numpy(), old) for v, old in zip(discriminator.weights, discriminator_before_d))
print("Discriminator changed; generator stayed unchanged.")

## Example 2

In [ ]:
z = Input(shape=(100,))
image = generator(z)
discriminator.trainable = False
validity = discriminator(image, training=False)
combined = Model(z, validity)
combined.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam())
assert {id(v) for v in combined.trainable_weights} == {id(v) for v in generator.trainable_weights}
print("Combined trainable parameters:", sum(int(np.prod(v.shape)) for v in combined.trainable_weights))
print("Frozen discriminator parameters:", discriminator.count_params())

## Example 3

In [ ]:
discriminator_before_g = [v.numpy().copy() for v in discriminator.weights]
generator_before_g = [v.numpy().copy() for v in generator.weights]
g_loss = combined.train_on_batch(noise, real_labels)
assert all(np.array_equal(v.numpy(), old) for v, old in zip(discriminator.weights, discriminator_before_g))
assert any(not np.array_equal(v.numpy(), old) for v, old in zip(generator.weights, generator_before_g))
print("Generator loss:", f"{float(g_loss):.6f}")
print("Generator changed; all discriminator weights and BatchNorm statistics stayed unchanged.")

## Independent regression checks

In [ ]:
assert np.isfinite(np.asarray(d_loss_real)).all() and np.isfinite(np.asarray(d_loss_fake)).all() and np.isfinite(g_loss)
for cycle in range(2):
    discriminator.trainable = True
    old_g = [v.numpy().copy() for v in generator.weights]
    old_d = [v.numpy().copy() for v in discriminator.weights]
    fake_batch = generator(noise, training=False).numpy()
    discriminator.train_on_batch(imgs, real_labels)
    discriminator.train_on_batch(fake_batch, fake_labels)
    assert all(np.array_equal(v.numpy(), old) for v, old in zip(generator.weights, old_g))
    assert any(not np.array_equal(v.numpy(), old) for v, old in zip(discriminator.weights, old_d))
    discriminator.trainable = False
    old_d = [v.numpy().copy() for v in discriminator.weights]
    old_g = [v.numpy().copy() for v in generator.weights]
    loss = combined.train_on_batch(noise, real_labels)
    assert np.isfinite(loss)
    assert all(np.array_equal(v.numpy(), old) for v, old in zip(discriminator.weights, old_d))
    assert any(not np.array_equal(v.numpy(), old) for v, old in zip(generator.weights, old_g))
    print("Alternating cycle", cycle + 1, ": discriminator-only and generator-only updates passed.")
assert int(discriminator.optimizer.iterations.numpy()) == 6
assert int(combined.optimizer.iterations.numpy()) == 3
assert not any(id(v) in {id(x) for x in discriminator.weights} for v in combined.trainable_weights)
print("All six discriminator and three generator updates verified, including frozen BatchNorm buffers.")

## Original source appendix

Full original blocks from raw source SHA-256 d2167867a2e9a06847208a00f739f8b540d690e4e73f22933733feba9843dad7. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
# Build and compile the discriminator
 discriminator = discriminator_model()
discriminator.compile(loss='binary_crossentropy',optimizer='adam', metrics=['accuracy'])
```

### Original block 2

```text
# Sample noise and generate a batch of new images
noise = np.random.normal(0, 1, (batch_size, 100))
gen_imgs = generator.predict(noise)
# Train the discriminator (real classified as ones and generated as zeros)
d_loss_real = discriminator.train_on_batch(imgs, valid)
d_loss_fake = discriminator.train_on_batch(gen_imgs, fake)
```

### Original block 3

```text
# Build the generator
 generator = generator_model()

 # The generator takes noise as input and generates image
 z = Input(shape=(100,))
 image = generator(z)
 
 # freeze the weights of the discriminator model
 discriminator.trainable = False
 
 # The discriminator takes generated images as input and determines validity
 valid = discriminator(img)
 
 # The combined model (stacked generator and discriminator)
 # Trains the generator to fool the discriminator
 combined = Model(z, valid)
```

### Original block 4

```text
combined.compile(loss='binary_crossentropy', optimizer=optimizer)
# Train the generator (wants discriminator to mistake images as real)
g_loss = self.combined.train_on_batch(noise, valid)
```